### Ingeniería de características y matrices de entrada del SVM

Construye las matrices de entrenamiento, validación y prueba que usa el SVM. Aplica la decisión de multicolinealidad de `01_Analisis_Limpieza_SVM.ipynb`, evalúa un conjunto de características derivadas (candidatas) y guarda el resultado.

**Entradas**

1. `data/processed/wednesday_low_and_slow_clean.csv`, generado por `limpieza_low_and_slow.ipynb`.
2. `reports/svm/columnas_multicolineales.json`, generado por `01_Analisis_Limpieza_SVM.ipynb`, con las columnas a eliminar y los parámetros de partición con que se calcularon.

**Salidas.** En `data/processed/SVM/<variante>/`: las matrices `X_train`, `X_val` y `X_test`, las etiquetas `y_*`, los metadatos `meta_*` (etiqueta original y bloque temporal) y `features.json`, que describe las características finales y las derivadas aceptadas o rechazadas.

**Decisiones de diseño**

1. Las candidatas se calculan fila a fila con columnas existentes, sin parámetros ajustados a los datos. Por eso se construyen igual en las tres particiones y no hay fuga de información. La decisión de incorporarlas se toma solo con entrenamiento.
2. Una candidata se acepta si cumple tres criterios: (a) no es casi constante; (b) no es redundante, es decir, su |Spearman| con cada característica original conservada no supera el umbral de la selección de multicolinealidad; y (c) su información mutua con la etiqueta llega a una fracción mínima de la máxima entre las originales conservadas.
3. Aquí no se escala ni se remuestrea. Ambos pasos dependen de estadísticos o ejemplos de entrenamiento y deben ejecutarse dentro de la validación cruzada para no filtrar información entre pliegues. Se hacen en `03_b_Train_Model.ipynb`.
4. `EXCLUIR_INIT_WIN` genera una variante sin las columnas `Init_Win_bytes_*` ni las candidatas que dependen de ellas, para el experimento de ablación. Cada variante se guarda en un directorio distinto.

**Contenido**

1. Librerías y parámetros
2. Lectura de la decisión de multicolinealidad
3. Carga y partición de los datos
4. Características candidatas
5. Evaluación de las candidatas con entrenamiento
6. Matrices finales
7. Vectores de características repetidos
8. Guardado

In [1]:
# Librerías y parámetros del cuaderno. Todo valor ajustable está en esta celda
# y se registra en features.json.

import sys
sys.path.append("../..")  # raíz del repositorio, desde notebooks/<carpeta>/

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

from src.preprocessing.clean import (
    cargar_dataset_limpio,
    train_test_split_by_flow,
    reportar_duplicados_de_features,
    COLUMNAS_A_ELIMINAR_CANDIDATAS,
    COLUMNAS_ATAJO_LABORATORIO,
    COLUMNAS_REDUNDANTES,
    COLUMNAS_CONSTANTES,
    HORA_INICIO_CAPTURA,
    HORA_FIN_CAPTURA,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Entradas
RUTA_CSV = "../../data/processed/wednesday_low_and_slow_clean.csv"
RUTA_DECISION = "../../reports/svm/columnas_multicolineales.json"

# True genera la variante sin Init_Win_bytes_* ni las derivadas que dependen de ellas
# (experimento de ablación)
EXCLUIR_INIT_WIN = False
COLUMNAS_INIT_WIN = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]
VARIANTE = "sin_init_win" if EXCLUIR_INIT_WIN else "completa"

# Se suma al denominador de las razones para evitar divisiones por cero
EPS = 1.0

# Una candidata debe tener al menos esta fracción de la información mutua máxima de las
# originales conservadas
MI_MINIMA_RELATIVA = 0.10

# Tamaño máximo de la muestra para estimar la información mutua
N_MUESTRA_MI = 50_000

# Salida: una carpeta por variante
OUT_DIR = Path("../../data/processed/SVM") / VARIANTE
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Variante: {VARIANTE}")

Variante: completa


**2. Lectura de la decisión de multicolinealidad**

Se lee el JSON de `01_Analisis_Limpieza_SVM.ipynb`: columnas a eliminar y conservar, umbrales de correlación y de casi constancia, semilla y parámetros de partición. Estos últimos se reutilizan en la sección siguiente para que la partición sea idéntica a la de esa decisión.

In [2]:
decision = json.loads(Path(RUTA_DECISION).read_text(encoding="utf-8"))

# Se reutilizan los parámetros de la decisión para que la partición sea idéntica
PARAMS_SPLIT = decision["params_split"]
UMBRAL_REDUNDANCIA = decision["umbral_spearman"]
UMBRAL_CASI_CONSTANTE = decision["umbral_casi_constante"]
RANDOM_STATE = decision["random_state"]

print(f"Decisión de multicolinealidad leída de: {RUTA_DECISION}")
print(f"Umbral de redundancia (|Spearman|):    {UMBRAL_REDUNDANCIA}")
print(f"Umbral de casi constancia:             {UMBRAL_CASI_CONSTANTE}")
print(f"Parámetros de la partición:            {PARAMS_SPLIT if PARAMS_SPLIT else 'valores predeterminados'}")
print(f"Columnas a eliminar:                   {len(decision['columnas_a_eliminar'])}")
print(f"Columnas originales conservadas:       {len(decision['columnas_conservadas'])}")

Decisión de multicolinealidad leída de: ../../reports/svm/columnas_multicolineales.json
Umbral de redundancia (|Spearman|):    0.98
Umbral de casi constancia:             0.999
Parámetros de la partición:            valores predeterminados
Columnas a eliminar:                   20
Columnas originales conservadas:       39


**3. Carga y partición de los datos**

Se carga el CSV limpio con `cargar_dataset_limpio()` y se comprueba que corresponde a la versión vigente de `clean.py` (reloj de 24 horas, sin las columnas que elimina la limpieza y sin nulos). Después se aplica la partición cronológica con los parámetros guardados en la decisión.

Por último se verifica la consistencia con `01_Analisis_Limpieza_SVM.ipynb`: número de filas del dataset y de entrenamiento, y coincidencia entre las columnas actuales y las que la decisión marca como conservadas o eliminadas. Una discrepancia indica que alguno de los dos archivos de entrada está desactualizado.

In [3]:
# El CSV debe corresponder a la versión vigente de clean.py
df = cargar_dataset_limpio(RUTA_CSV)

# Timestamp en tipo fecha y reloj de 24 horas (debe haber tráfico de la tarde)
assert pd.api.types.is_datetime64_any_dtype(df["Timestamp"]), "Timestamp no es de tipo fecha"
horas = df["Timestamp"].dt.hour
assert horas.between(HORA_INICIO_CAPTURA, HORA_FIN_CAPTURA).all(), "Horas fuera del horario de captura"
assert horas.max() >= 13, "El CSV no contiene la corrección del reloj de 12 horas"

# Sin las columnas que elimina la limpieza
eliminadas = (COLUMNAS_A_ELIMINAR_CANDIDATAS + COLUMNAS_ATAJO_LABORATORIO
              + COLUMNAS_REDUNDANTES + COLUMNAS_CONSTANTES)
presentes = [c for c in eliminadas if c in df.columns]
assert not presentes, f"Columnas que la limpieza debió eliminar siguen en el CSV: {presentes}"

# Sin nulos
assert not df.isna().any().any(), "Existen valores nulos en el CSV"

n_filas_dataset = len(df)
print(f"CSV cargado: {n_filas_dataset:,} filas x {df.shape[1]} columnas")

# Partición cronológica con los parámetros de la decisión
(X_train, X_val, X_test, y_train, y_val, y_test,
 meta_train, meta_val, meta_test) = train_test_split_by_flow(df, return_meta=True, **PARAMS_SPLIT)
del df

# Características = columnas sin los metadatos de la partición
COLUMNAS_METADATA = ["Timestamp", "bloque"]
features_actuales = [c for c in X_train.columns if c not in COLUMNAS_METADATA]

# Consistencia con la decisión de multicolinealidad (si falla, algún archivo está desactualizado)
assert n_filas_dataset == decision["n_filas_dataset"], (
    "El número de filas del CSV no coincide con el de la decisión; ejecutar de nuevo Analisis_Limpieza_SVM.ipynb")
assert len(X_train) == decision["n_train"], (
    "El tamaño de la partición de entrenamiento no coincide con el de la decisión")
esperadas = set(decision["columnas_conservadas"]) | set(decision["columnas_a_eliminar"])
assert set(features_actuales) == esperadas, (
    f"Las columnas actuales difieren de las de la decisión: {sorted(set(features_actuales) ^ esperadas)}")
print("\nConsistencia con la decisión de multicolinealidad verificada.")

CSV cargado: 443,614 filas x 62 columnas
Fronteras de bloque (outlier_quantile=0.0, desempate=csv):
  Slowloris:    [2017-07-05 09:48:00 .. 2017-07-05 10:11:00]  (5790 filas)
  Slowhttptest: [2017-07-05 10:15:00 .. 2017-07-05 10:37:00]  (5499 filas)

Composición por bloque:
  Bloque 1: 53375 filas | Timestamp [2017-07-05 08:42:00 .. 2017-07-05 09:47:00] | ataques: 0 (0.00%)
  Bloque 2: 30362 filas | Timestamp [2017-07-05 09:48:00 .. 2017-07-05 10:11:00] | ataques: 5790 (19.07%)
  Bloque 3: 3578 filas | Timestamp [2017-07-05 10:12:00 .. 2017-07-05 10:14:00] | ataques: 0 (0.00%)
  Bloque 4: 33203 filas | Timestamp [2017-07-05 10:15:00 .. 2017-07-05 10:37:00] | ataques: 5499 (16.56%)
  Bloque 5: 323096 filas | Timestamp [2017-07-05 10:38:00 .. 2017-07-05 17:10:00] | ataques: 0 (0.00%)

Train: 310529 filas | Val: 66541 filas | Test: 66544 filas
Proporción de ataques -> train: 0.0265, val: 0.0283, test: 0.0177

Consistencia con la decisión de multicolinealidad verificada.


**4. Características candidatas**

Se definen cinco características derivadas, cada una con su fórmula, las columnas de las que depende y la razón de incluirla. Se calculan con **todas** las columnas originales, antes de eliminar las redundantes, porque una derivada puede depender de una columna que la selección de multicolinealidad descartó.

La transformación logarítmica con signo se aplica más adelante, al escalar, así que las candidatas se guardan sin transformar.

In [4]:
# Candidatas: fórmula, columnas de las que dependen, justificación y función de cálculo.
# Cada función opera fila a fila y no ajusta nada con los datos
DEFINICIONES = {
    "ratio_active_idle": {
        "formula": "Active Mean / (Idle Std + EPS)",
        "componentes": ["Active Mean", "Idle Std"],
        "justificacion": "Relación entre el tiempo medio de actividad y la variabilidad de los periodos "
                         "de inactividad; busca reflejar conexiones abiertas con actividad escasa.",
        "funcion": lambda X: X["Active Mean"] / (X["Idle Std"] + EPS),
    },
    "ratio_fwd_bwd_pps": {
        "formula": "Fwd Packets/s / (Bwd Packets/s + EPS)",
        "componentes": ["Fwd Packets/s", "Bwd Packets/s"],
        "justificacion": "Asimetría entre la tasa de paquetes hacia el servidor y la de respuesta; "
                         "busca reflejar un servidor que responde poco.",
        "funcion": lambda X: X["Fwd Packets/s"] / (X["Bwd Packets/s"] + EPS),
    },
    "low_and_slow_score": {
        "formula": "Flow Duration / (Flow Bytes/s + EPS)",
        "componentes": ["Flow Duration", "Flow Bytes/s"],
        "justificacion": "Cociente entre la duración del flujo y su tasa de bytes; valores elevados "
                         "corresponden a conexiones prolongadas de bajo caudal.",
        "funcion": lambda X: X["Flow Duration"] / (X["Flow Bytes/s"] + EPS),
    },
    "payload_eficiencia_fwd": {
        "formula": "Total Length of Fwd Packets / (Fwd Packets/s * Flow Duration + EPS)",
        "componentes": ["Total Length of Fwd Packets", "Fwd Packets/s", "Flow Duration"],
        "justificacion": "Bytes de carga enviados por paquete hacia el servidor, estimados a partir de "
                         "la tasa de paquetes y la duración del flujo.",
        "funcion": lambda X: X["Total Length of Fwd Packets"] / (X["Fwd Packets/s"] * X["Flow Duration"] + EPS),
    },
    "ventana_inicial_relativa": {
        "formula": "Init_Win_bytes_forward / (Flow Duration + EPS)",
        "componentes": ["Init_Win_bytes_forward", "Flow Duration"],
        "justificacion": "Tamaño inicial de la ventana TCP hacia el servidor relativo a la duración del flujo.",
        "funcion": lambda X: X["Init_Win_bytes_forward"] / (X["Flow Duration"] + EPS),
    },
}

# Las columnas de las que dependen deben existir
for nombre, d in DEFINICIONES.items():
    faltan = [c for c in d["componentes"] if c not in features_actuales]
    assert not faltan, f"{nombre}: columnas ausentes {faltan}"

# En la variante de ablación se omiten las que dependen de Init_Win_bytes_*
no_aplicables = []
if EXCLUIR_INIT_WIN:
    no_aplicables = [n for n, d in DEFINICIONES.items() if set(d["componentes"]) & set(COLUMNAS_INIT_WIN)]
candidatas = [n for n in DEFINICIONES if n not in no_aplicables]
print(f"Candidatas a evaluar: {candidatas}")
if no_aplicables:
    print(f"Candidatas omitidas en la variante '{VARIANTE}': {no_aplicables}")


def construir_candidatas(X):
    """Devuelve las candidatas calculadas con las columnas originales de X, con el mismo índice."""
    return pd.DataFrame({n: DEFINICIONES[n]["funcion"](X) for n in candidatas}, index=X.index)


# Misma función en las tres particiones
C_train = construir_candidatas(X_train)
C_val = construir_candidatas(X_val)
C_test = construir_candidatas(X_test)

# Los cocientes no deben producir nulos ni infinitos
for nombre, C in [("entrenamiento", C_train), ("validación", C_val), ("prueba", C_test)]:
    assert np.isfinite(C.to_numpy(dtype=float)).all(), f"Valores no finitos en las candidatas de {nombre}"
print("Candidatas construidas sin valores nulos ni infinitos en las tres particiones.")

Candidatas a evaluar: ['ratio_active_idle', 'ratio_fwd_bwd_pps', 'low_and_slow_score', 'payload_eficiencia_fwd', 'ventana_inicial_relativa']
Candidatas construidas sin valores nulos ni infinitos en las tres particiones.


**5. Evaluación de las candidatas con entrenamiento**

Cada candidata se evalúa con tres criterios:

- **Casi constancia:** proporción de filas que ocupa su valor más frecuente, frente a `UMBRAL_CASI_CONSTANTE`.
- **Redundancia:** máximo |Spearman| con las características originales conservadas, frente a `UMBRAL_REDUNDANCIA`.
- **Relevancia:** información mutua con la etiqueta, frente a `MI_MINIMA_RELATIVA` veces la máxima entre las originales conservadas. Se estima igual que en `01_Analisis_Limpieza_SVM.ipynb` (muestra estratificada, escala logarítmica con signo y columnas con diez valores distintos o menos como discretas), con candidatas y originales sobre la misma muestra.

La aceptación es el resultado de aplicar los criterios; no se ajustan para obtener un resultado en particular.

In [5]:
# Originales conservadas por la decisión de multicolinealidad (sin Init_Win_bytes_* en la
# variante de ablación)
originales = list(decision["columnas_conservadas"])
if EXCLUIR_INIT_WIN:
    originales = [c for c in originales if c not in COLUMNAS_INIT_WIN]

# (a) Casi constancia
dominante = C_train.apply(lambda s: s.value_counts(normalize=True).iloc[0])
casi_constante = dominante >= UMBRAL_CASI_CONSTANTE

# (b) Redundancia: Spearman (Pearson sobre rangos) contra cada original conservada
rangos_originales = X_train[originales].rank()
rangos_candidatas = C_train.rank()
corr = pd.DataFrame({c: rangos_originales.corrwith(rangos_candidatas[c]) for c in candidatas}).fillna(0.0)
max_rho = corr.abs().max()
pareja = corr.abs().idxmax()
redundante = max_rho > UMBRAL_REDUNDANCIA


def informacion_mutua(X, y, n_max=N_MUESTRA_MI, random_state=RANDOM_STATE):
    """Información mutua (nats) de cada columna de X con y, sobre una muestra estratificada.

    Usa el mismo procedimiento que 01_Analisis_Limpieza_SVM.ipynb para que los valores sean comparables.
    """
    if n_max < len(X):
        X_s, _, y_s, _ = train_test_split(X, y, train_size=n_max, stratify=y, random_state=random_state)
    else:
        X_s, y_s = X, y
    X_log = np.sign(X_s) * np.log1p(np.abs(X_s))
    discretas = (X_s.nunique() <= 10).to_numpy()
    X_log.loc[:, discretas] = X_s.loc[:, discretas]
    return pd.Series(
        mutual_info_classif(X_log, y_s, discrete_features=discretas, random_state=random_state),
        index=X.columns,
    )


# (c) Relevancia: originales y candidatas se estiman juntas, sobre la misma muestra
mi = informacion_mutua(pd.concat([X_train[originales], C_train], axis=1), y_train)
mi_referencia = mi[originales].max()
mi_requerida = MI_MINIMA_RELATIVA * mi_referencia
irrelevante = mi[candidatas] < mi_requerida

# Resultado de la evaluación y decisión de aceptación
evaluacion = pd.DataFrame({
    "proporcion_dominante": dominante,
    "max_abs_rho": max_rho,
    "redundante_con": pareja,
    "info_mutua": mi[candidatas],
    "info_mutua_requerida": mi_requerida,
})
evaluacion["casi_constante"] = casi_constante
evaluacion["redundante"] = redundante
evaluacion["irrelevante"] = irrelevante
evaluacion["aceptada"] = ~(casi_constante | redundante | irrelevante)

def motivo(fila):
    if fila["casi_constante"]:
        return "casi constante"
    if fila["redundante"]:
        return f"redundante con {fila['redundante_con']}"
    if fila["irrelevante"]:
        return "información mutua inferior a la mínima requerida"
    return ""

evaluacion["motivo_rechazo"] = evaluacion.apply(motivo, axis=1)
display(evaluacion.round(4))

aceptadas = evaluacion.index[evaluacion["aceptada"]].tolist()
rechazadas = evaluacion.index[~evaluacion["aceptada"]].tolist()
print(f"\nInformación mutua máxima de las originales conservadas: {mi_referencia:.4f}")
print(f"Candidatas aceptadas:  {aceptadas if aceptadas else 'ninguna'}")
print(f"Candidatas rechazadas: {rechazadas if rechazadas else 'ninguna'}")

,proporcion_dominante,max_abs_rho,redundante_con,info_mutua,info_mutua_requerida,casi_constante,redundante,irrelevante,aceptada,motivo_rechazo
ratio_active_idle,0.8245,0.9837,Idle Max,0.0593,0.0099,False,True,False,False,redundante con Idle Max
ratio_fwd_bwd_pps,0.0276,0.8386,Flow Packets/s,0.0925,0.0099,False,False,False,True,
low_and_slow_score,0.0142,0.9174,Flow Bytes/s,0.0944,0.0099,False,False,False,True,
payload_eficiencia_fwd,0.1036,0.9990,Fwd Packet Length Mean,0.0744,0.0099,False,True,False,False,redundante con Fwd Packet Length Mean
ventana_inicial_relativa,0.0184,0.7606,Init_Win_bytes_forward,0.1000,0.0099,False,False,False,True,



Información mutua máxima de las originales conservadas: 0.0992
Candidatas aceptadas:  ['ratio_fwd_bwd_pps', 'low_and_slow_score', 'ventana_inicial_relativa']
Candidatas rechazadas: ['ratio_active_idle', 'payload_eficiencia_fwd']


**6. Matrices finales**

Cada matriz reúne las originales conservadas (sin `Init_Win_bytes_*` en la variante de ablación) y las candidatas aceptadas. Los metadatos (etiqueta original y bloque) van en tablas aparte para que no se usen como características por error. Se verifica que las columnas coincidan entre particiones y que no haya nulos, infinitos, columnas duplicadas ni metadatos entre las características.

In [6]:
# Matrices finales: originales conservadas + candidatas aceptadas
def ensamblar(X, C):
    return pd.concat([X[originales], C[aceptadas]], axis=1)

X_train_f = ensamblar(X_train, C_train)
X_val_f = ensamblar(X_val, C_val)
X_test_f = ensamblar(X_test, C_test)

# Metadatos fuera de la matriz de características: etiqueta original y bloque
def metadatos(meta, X):
    return pd.DataFrame({"Label": meta["Label"], "bloque": X["bloque"]})

meta_train_f = metadatos(meta_train, X_train)
meta_val_f = metadatos(meta_val, X_val)
meta_test_f = metadatos(meta_test, X_test)

# Coherencia de las matrices finales
columnas_finales = list(X_train_f.columns)
for nombre, X, y, m in [("entrenamiento", X_train_f, y_train, meta_train_f),
                        ("validación", X_val_f, y_val, meta_val_f),
                        ("prueba", X_test_f, y_test, meta_test_f)]:
    assert list(X.columns) == columnas_finales, f"Las columnas de {nombre} difieren de las de entrenamiento"
    assert X.columns.is_unique, f"Hay columnas duplicadas en {nombre}"
    assert not X.isna().any().any(), f"Hay valores nulos en {nombre}"
    assert np.isfinite(X.to_numpy(dtype=float)).all(), f"Hay valores no finitos en {nombre}"
    assert len(X) == len(y) == len(m), f"Longitudes inconsistentes en {nombre}"
    assert (X.index == y.index).all() and (X.index == m.index).all(), f"Índices inconsistentes en {nombre}"
assert not {"Timestamp", "bloque", "Label", "label_binaria"} & set(columnas_finales), "Metadatos entre las características"

print(f"Características finales: {len(columnas_finales)} "
      f"({len(originales)} originales + {len(aceptadas)} derivadas)")
for nombre, X in [("entrenamiento", X_train_f), ("validación", X_val_f), ("prueba", X_test_f)]:
    print(f"  {nombre}: {X.shape}")

Características finales: 42 (39 originales + 3 derivadas)
  entrenamiento: (310529, 42)
  validación: (66541, 42)
  prueba: (66544, 42)


**7. Vectores de características repetidos**

Para el SVM, dos flujos con el mismo vector son el mismo ejemplo. Por partición y clase se cuenta cuántos vectores se repiten dentro de la partición y cuántos de validación y prueba ya aparecen en entrenamiento, además de los que aparecen con ambas etiquetas. No se elimina ninguna fila: solo se documenta la magnitud del fenómeno, que influye en cómo se interpretan las métricas.

In [7]:
# Vectores repetidos sobre las matrices finales (sin metadatos)
reporte_duplicados = reportar_duplicados_de_features(
    X_train_f, X_val_f, X_test_f, y_train, y_val, y_test, columnas_metadata=()
)

split  clase  filas  dup_en_split  en_train
train      0 302299         29608       NaN
train      1   8230           290       NaN
  val      0  64657          4812    7858.0
  val      1   1884           181     121.0
 test      0  65369          4147    8106.0
 test      1   1175           109      33.0

Filas cuyo vector aparece con ambas etiquetas: 0


In [8]:
# Filas cuyo vector aparece con ambas etiquetas, por clase (vacío = ninguna)
y_todo = pd.concat([y_train, y_val, y_test]).to_numpy()
h = pd.concat([pd.util.hash_pandas_object(X, index=False)
               for X in (X_train_f, X_val_f, X_test_f)]).to_numpy()
tabla_h = pd.DataFrame({"h": h, "y": y_todo})
ambiguas = tabla_h.groupby("h")["y"].transform("nunique") > 1
print(tabla_h.loc[ambiguas, "y"].value_counts())

Series([], Name: count, dtype: int64)


**8. Guardado**

En el directorio de la variante se guardan las matrices, las etiquetas y los metadatos de cada partición, junto con `features.json`, que registra las características finales, las derivadas aceptadas y rechazadas (con fórmula y motivo) y los parámetros usados. Al terminar se comprueba que filas y columnas de cada archivo coinciden con los datos en memoria.

In [9]:
particiones = {
    "train": (X_train_f, y_train, meta_train_f),
    "val": (X_val_f, y_val, meta_val_f),
    "test": (X_test_f, y_test, meta_test_f),
}
for nombre, (X, y, m) in particiones.items():
    X.to_csv(OUT_DIR / f"X_{nombre}.csv", index=False)
    y.rename("label_binaria").to_csv(OUT_DIR / f"y_{nombre}.csv", index=False, header=True)
    m.to_csv(OUT_DIR / f"meta_{nombre}.csv", index=False)

# Características y parámetros usados
features_json = {
    "variante": VARIANTE,
    "excluir_init_win": EXCLUIR_INIT_WIN,
    "escalado": "no aplicado; se realiza en Train_Model.ipynb dentro de la validación cruzada",
    "remuestreo": "no aplicado",
    "fuente_datos": RUTA_CSV,
    "fuente_decision": RUTA_DECISION,
    "params_split": PARAMS_SPLIT,
    "umbral_redundancia": UMBRAL_REDUNDANCIA,
    "umbral_casi_constante": UMBRAL_CASI_CONSTANTE,
    "mi_minima_relativa": MI_MINIMA_RELATIVA,
    "eps": EPS,
    "random_state": RANDOM_STATE,
    "n_train": len(X_train_f), "n_val": len(X_val_f), "n_test": len(X_test_f),
    "caracteristicas_finales": columnas_finales,
    "originales_conservadas": originales,
    "derivadas_aceptadas": {
        n: {"formula": DEFINICIONES[n]["formula"], "componentes": DEFINICIONES[n]["componentes"]}
        for n in aceptadas
    },
    "derivadas_rechazadas": {
        n: {"formula": DEFINICIONES[n]["formula"], "motivo": evaluacion.loc[n, "motivo_rechazo"]}
        for n in rechazadas
    },
    "derivadas_no_aplicables": no_aplicables,
}
(OUT_DIR / "features.json").write_text(json.dumps(features_json, indent=2, ensure_ascii=False), encoding="utf-8")

# Los archivos escritos deben coincidir con los datos en memoria
def filas_archivo(ruta):
    with open(ruta, encoding="utf-8") as f:
        return sum(1 for _ in f) - 1  # sin el encabezado

for nombre, (X, y, m) in particiones.items():
    ruta = OUT_DIR / f"X_{nombre}.csv"
    columnas = pd.read_csv(ruta, nrows=0).columns.tolist()
    assert columnas == list(X.columns), f"Columnas de X_{nombre}.csv difieren de las de memoria"
    assert filas_archivo(ruta) == len(X), f"Filas de X_{nombre}.csv difieren de las de memoria"
    assert filas_archivo(OUT_DIR / f"y_{nombre}.csv") == len(y)
    assert filas_archivo(OUT_DIR / f"meta_{nombre}.csv") == len(m)
    print(f"  {nombre}: {len(X):,} filas x {X.shape[1]} columnas | {ruta.stat().st_size / 1024**2:.1f} MB")
print(f"\nArchivos guardados en: {OUT_DIR.resolve()}")

  train: 310,529 filas x 42 columnas | 74.7 MB
  val: 66,541 filas x 42 columnas | 16.0 MB
  test: 66,544 filas x 42 columnas | 15.9 MB

Archivos guardados en: C:\Users\juans\OneDrive\Documents\Github\low-and-slow-ids\data\processed\SVM\completa


In [10]:
# Resumen: composición de las características finales y de las particiones
print(f"Variante: {VARIANTE}")
print(f"Características originales conservadas: {len(originales)}")
print(f"Características derivadas aceptadas:    {len(aceptadas)} {aceptadas if aceptadas else ''}")
print(f"Características derivadas rechazadas:   {len(rechazadas)}")
print(f"Características finales:               {len(columnas_finales)}")
print(f"\nParticiones (filas | proporción de ataques):")
for nombre, (X, y, m) in particiones.items():
    print(f"  {nombre:6s}: {len(X):>8,} | {y.mean():.4%}")

Variante: completa
Características originales conservadas: 39
Características derivadas aceptadas:    3 ['ratio_fwd_bwd_pps', 'low_and_slow_score', 'ventana_inicial_relativa']
Características derivadas rechazadas:   2
Características finales:               42

Particiones (filas | proporción de ataques):
  train :  310,529 | 2.6503%
  val   :   66,541 | 2.8313%
  test  :   66,544 | 1.7657%


**Uso de los resultados**

- `03_b_Train_Model.ipynb` carga las matrices de `data/processed/SVM/<variante>/` y debe aplicar, dentro del `Pipeline` que se ajusta en la validación cruzada, la transformación logarítmica con signo y la estandarización. Los metadatos `meta_*` permiten calcular el desempeño por tipo de ataque y por bloque.
- La variante `sin_init_win` se genera corriendo de nuevo este cuaderno con `EXCLUIR_INIT_WIN = True`. Compararla con `completa` es el experimento de ablación.
- Las matrices dependen de `clean.py`, del CSV procesado y de `columnas_multicolineales.json`. Si cualquiera cambia, hay que correr de nuevo, en orden, `limpieza_low_and_slow.ipynb`, `01_Analisis_Limpieza_SVM.ipynb` y este cuaderno.
- Los archivos `X_train.csv`, `X_val.csv`, `X_test.csv`, `*_pre_smote.csv` y `svm_features_reducidas.csv` que están directamente en `data/processed/SVM/` son de una versión anterior del procedimiento y no deben usarse.